In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
df.Category.value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [6]:
df['spam'] = df['Category'].apply(lambda x: 1 if x =='spam' else 0)

In [7]:
df.shape

(5572, 3)

In [8]:
df.head()

,Category,Message,spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.Message, df.spam, test_size=0.2)

In [10]:
X_train.shape

(4457,)

In [11]:
X_test.shape

(1115,)

In [12]:
type(X_train)

pandas.core.series.Series

In [17]:
X_train[:4]

4580                   No plans yet. What are you doing ?
5412    Gr8. So how do you handle the victoria island ...
1268     SERIOUSLY. TELL HER THOSE EXACT WORDS RIGHT NOW.
2668                   Why de. You looking good only:-)..
Name: Message, dtype: object

In [14]:
type(y_train)

pandas.core.series.Series

In [15]:
y_train[:4]

4580    0
5412    0
1268    0
2668    0
Name: spam, dtype: int64

### Create bag of words representation using CountVectorizer

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()

X_train_cv = v.fit_transform(X_train.values)
X_train_cv

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 59286 stored elements and shape (4457, 7669)>

In [20]:
X_train_cv.toarray()[:2][0]

array([0, 0, 0, ..., 0, 0, 0], shape=(7669,))

In [21]:
X_train_cv.shape

(4457, 7669)

In [22]:
v.get_feature_names_out()[1771]

'chat'

In [23]:
v.vocabulary_

{'no': 4765,
 'plans': 5211,
 'yet': 7624,
 'what': 7402,
 'are': 1068,
 'you': 7637,
 'doing': 2420,
 'gr8': 3207,
 'so': 6236,
 'how': 3507,
 'do': 2400,
 'handle': 3307,
 'the': 6747,
 'victoria': 7211,
 'island': 3728,
 'traffic': 6936,
 'plus': 5233,
 'when': 7408,
 'album': 915,
 'due': 2501,
 'seriously': 5981,
 'tell': 6691,
 'her': 3402,
 'those': 6792,
 'exact': 2703,
 'words': 7520,
 'right': 5738,
 'now': 4812,
 'why': 7429,
 'de': 2211,
 'looking': 4179,
 'good': 3178,
 'only': 4917,
 'hai': 3291,
 'priya': 5383,
 'doctor': 2405,
 'said': 5841,
 'pa': 5013,
 'where': 7411,
 'sian': 6101,
 'aft': 870,
 'meeting': 4400,
 'supervisor': 6561,
 'got': 3196,
 'work': 7521,
 'liao': 4069,
 'working': 7525,
 'going': 3168,
 'out': 4983,
 'to': 6864,
 'buy': 1606,
 'mum': 4620,
 'present': 5353,
 'ar': 1063,
 'hmm': 3438,
 'too': 6902,
 'many': 4324,
 'of': 4865,
 'them': 6754,
 'unfortunately': 7082,
 'pics': 5175,
 'obviously': 4855,
 'arent': 1071,
 'hot': 3495,
 'cakes': 1626,


In [25]:
X_train_np = X_train_cv.toarray()
X_train_np[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(7669,))

In [26]:
np.where(X_train_np[0]!=0)

(array([1068, 2420, 4765, 5211, 7402, 7624, 7637]),)

In [31]:
X_train[:4][2668]

'Why de. You looking good only:-)..'

In [32]:
X_train_np[0][1771]

np.int64(0)

## Train the naive bayes model

In [33]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_cv, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [35]:
X_test_cv = v.transform(X_test)

In [36]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_cv)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       976
           1       0.99      0.91      0.95       139

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [37]:
emails = [
    'Hey mohan, can we get together to watch footbal game tomorrow?',
    'Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!'
]

emails_count = v.transform(emails)
model.predict(emails_count)

array([0, 1])

## Train the model using sklearn pipeline and reduce number of lines of code

In [39]:
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

In [40]:
clf.fit(X_train, y_train)

,steps,"[('vectorizer', ...), ('nb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [41]:

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       976
           1       0.99      0.91      0.95       139

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

